# 04 - Gold Exploration with PySpark SQL

Run business-facing SQL over the latest gold layer. Open the Streamlit app for interactive charts.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.spark_pipeline import create_spark_session, load_latest_gold
from insightsstream.storage import create_store

spark = create_spark_session("InsightsStream-Notebook-Explore")
gold, run = load_latest_gold(spark=spark, store=create_store())
gold.createOrReplaceTempView("gold_daily_metrics")

In [ ]:
spark.sql("""
    SELECT
        SUM(event_count) AS total_events,
        ROUND(SUM(total_value), 2) AS total_value,
        MIN(event_date) AS first_event_date,
        MAX(event_date) AS latest_event_date
    FROM gold_daily_metrics
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT
        event_date,
        metric,
        total_value,
        DENSE_RANK() OVER (
            PARTITION BY event_date ORDER BY total_value DESC
        ) AS value_rank
    FROM gold_daily_metrics
    ORDER BY event_date, value_rank
""").show(100, truncate=False)